# FLAG · RF-E2E — pipeline Evaluation của hệ chính (RF-L2-092, CB dev 18,61)

**Một lượt, từ ảnh/audio gốc ra `submission.zip`. Mọi cặp được chấm độc lập (mức 0): không chuẩn hoá điểm, không cohort, không TTA.**

**Hệ chính:**
- `t3`: projector QV-Whisper, học trên người ngoài (5 930 người VoxCeleb2, 24 người MAV-Celeb v1/v2 và 7 713 người VoxBlink, đều đã lọc Bengali theo người; RF-L2-088 = RF-L2-066 train lại không có 59 người VoxCeleb2 bị cờ ở video khác) + người v4 train (thêm batch v4 cùng giới), trên đặc trưng [CLIP ‖ ImageBind-mặt] × [SimAM-ResNet100 ‖ ResNet221 ‖ ECAPA ‖ Whisper large-v3 encoder (để nguyên)]; điểm = cosine;
- `ib`: ImageBind-mặt × ImageBind-giọng có LoRA C1c (LoRA train trên VoxCeleb2 + v1/v2 đã lọc Bengali, không có 59 người trên; 19 122 clip / 5 929 người);
- `nf`, `nv` (RF-L2-092, QF): log độ dài vector mặt / giọng sau projector, trước chuẩn hóa L2 (trung bình 3 seed) — thuộc tính riêng của từng file;
- `age` (chỉ file no_gender): −|tuổi mặt − b(tuổi giọng)|, ViT-age (nateraw) và audeering wav2vec2 age-gender để nguyên, b = 2 tham số học trên v4 train (tiếng Anh).
- File **no_gender**: CA-FUSE trên (`t3`, `ib`, `age`, `nf`, `nv`) + điều kiện giới của cặp + tích; file **gender**: hồi quy logistic trên (`t3`, `ib`, `nf`, `nv`). Chọn theo nhãn protocol của file do BTC đặt.

**Mọi thứ đã học đều cố định trong dataset artifact** (bản Private, khoảng 30 MB; không chứa dữ liệu train hay tập tham chiếu nào):
- `final_params.npz`: probe giới (chuẩn hoá + PCA, hệ số logistic), trung bình ImageBind mặt của train, hằng số điền NaN, ánh xạ tuổi `b`, hai bộ trộn (hệ số);
- LoRA C1c (tên file `lora_C1.pt`);
- projector QV-Whisper (tên file `projector_Q.pt`).

Notebook **không fit gì trên file cần chấm**. Nó chỉ trích embedding của từng file rồi chấm từng cặp.

## Trước khi chạy

1. **GPU T4**, **Internet: On**.
2. Add Input:
   - dataset FLAG (dev hoặc Evaluation);
   - dataset artifact `flag-rf-final-066` (Private; version mới nhất = thư mục `_tam/final_release_092` + `vgg_face.pt`, vì máy chủ Oxford của VGG-Face có lúc không trả lời).
3. **Save & Run All.** Dev mất khoảng 1,5 giờ (thêm Whisper large-v3 và hai mô hình tuổi, tải từ Hugging Face).
4. **Kiểm khi chạy trên dev:** notebook so với ZIP đã nộp (`submission_reference.zip` trong artifact = bài 18,61). Cần tương quan hạng ≥ 0,999 ở cả 4 file.
5. **Khi chạy Evaluation:**
   - nếu tên file khác dev, sửa `NAMES` ở ô cấu hình;
   - lượt nộp đầu: nếu EER > 50 thì đảo dấu điểm.

In [ ]:
print("[boot] FLAG_163i_rfs_infer | python", __import__("sys").version.split()[0], "| start", __import__("time").strftime("%Y-%m-%d %H:%M:%S"), flush=True)  # heartbeat rule: first output, before any import
print("[cell] config", flush=True)
from pathlib import Path

DATA_ROOT = Path('/kaggle/input/datasets/ngnnguynchn/flag2027-evalphase-p')   # Evaluation lists (patch_163i_eval.py)
ART = None           # None = tự tìm thư mục có manifest_final.json
REF_ZIP = None       # None = file submission_*.zip trong ART nếu có (chỉ để kiểm trên dev)
USE_CSV = True       # dùng đặc trưng VGG / ECAPA do BTC cung cấp nếu có (đúng như bài đã nộp); không có thì tự trích
SMOKE_PAIRS = None   # chỉ dùng khi chạy thử ở máy
NAMES = None         # None = 4 file như dev; Evaluation: {(protocol, lang): tên file trong zip nộp}
WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT = WORK_ROOT / 'FLAG_RF_E2E'
IB_CKPT = None; R221_CKPT = None; VGG_CKPT = None; VB2_CKPT = None   # None = tải
URL = {'imagebind_huge': 'https://huggingface.co/PKU-Alignment/AnyRewardModel/resolve/40c7e2ade237848a194b2b77517da024c4ec8532/imagebind/imagebind_huge.pth',
       'resnet221_lm': 'https://huggingface.co/Wespeaker/wespeaker-voxceleb-resnet221-LM/resolve/main/voxceleb_resnet221_LM.pt',
       'vgg_face': 'https://www.robots.ox.ac.uk/~albanie/models/pytorch-mcn/vgg_face_dag.pth',
       'voxblink2_samresnet100_ft': 'https://huggingface.co/MCplayer/voxblink2_samresnet100_ft/resolve/main/models/voxblink2_samresnet100_ft/avg_model.pt'}
SHA = {'imagebind_huge': 'd6f6c22bedcc90708448d5d2fbb7b2db9c73f505dc89bd0b2e09b23af1b62157',
       'resnet221_lm': '47d76239f1e865b273e31e30f691959061ff6565ed6c4e7be9639a65d9662eb5',
       'vgg_face': 'f070b241e3faf17a08c78f8700d334413230bac5de7fbcfd01b3cf05ce10de1a',
       'voxblink2_samresnet100_ft': '5aeee438ca23c0ca6e341bab6c6bf7f465497e1dc323bb1bc1074d6a0c778b11'}
print('Output:', OUT)

In [ ]:
print("[cell] setup+core+checkpoints", flush=True)
import importlib.util, subprocess, sys, json, hashlib, urllib.request, os, shutil, zipfile, time
def pip(*a): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a], check=True)
if importlib.util.find_spec('imagebind') is None:
    pip('--no-deps', 'git+https://github.com/facebookresearch/ImageBind.git')
for mod in ('ftfy', 'wcwidth', 'regex', 'portalocker', 'iopath', 'einops', 'timm', 'av', 'hyperpyyaml', 'ruamel.yaml'):
    if importlib.util.find_spec(mod.split('.')[0]) is None:
        pip(mod) if mod == 'av' else pip('--no-deps', mod)
import importlib.metadata as md_
if tuple(int(x) for x in md_.version('ruamel.yaml').split('.')[:2]) >= (0, 19):
    pip('ruamel.yaml<0.19')
try:   # SpeechBrain ECAPA (the BTC-space voice encoder); not in the Kaggle image
    sb_ok = tuple(int(x) for x in md_.version('speechbrain').split('.')[:3]) >= (1, 1, 1)
except md_.PackageNotFoundError:
    sb_ok = False
if not sb_ok:
    pip('--no-deps', '-U', 'speechbrain>=1.1.1')
OUT.mkdir(parents=True, exist_ok=True)

CORE_SOURCE = '"""FLAG 2027 face-voice association: the submitted system (RF-L2-059), generated by make_final_core.py from\nflag_e2e_core.py (sha256 276d19faded81268). Feature extractors, data loading and FinalScorer only."""\nfrom pathlib import Path\nimport hashlib, json, platform, shutil, time, zipfile, os\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom sklearn.metrics import roc_curve\nfrom threadpoolctl import threadpool_limits\nfrom PIL import Image\nVERSION = "RF-L0-002-v1"\nDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nCELLS = {("no_gender", "English"): "no_gender/sub_score_v4_English_heard.txt",\n         ("no_gender", "Bangla"): "no_gender/sub_score_v4_Bangla_unheard.txt",\n         ("gender", "English"): "gender/sub_score_v4_English_heard.txt",\n         ("gender", "Bangla"): "gender/sub_score_v4_Bangla_unheard.txt"}\nEXPECTED_ROWS = {("no_gender", "English"):1008, ("no_gender", "Bangla"):1406,\n                 ("gender", "English"):982, ("gender", "Bangla"):1468}\ndef table(path, names):\n    t = pd.read_csv(path, sep=r\'\\s+\', header=None, dtype=str, keep_default_na=False)\n    if t.shape[1] != len(names):\n        raise ValueError(f\'{path}: expected {len(names)} columns, got {t.shape[1]}\')\n    t.columns = names\n    if t.pair_id.duplicated().any():\n        raise ValueError(f\'Duplicate pair_id in {path}\')\n    return t\ndef write_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    temp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False), encoding=\'utf-8\')\n    temp.replace(path)\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\ndef file_hash(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for b in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(b)\n    return h.hexdigest()\nVERSION = "RF-L0-003-v1"\nMEAN_RGB = np.array([129.186279296875, 104.76238250732422, 93.59396362304688], dtype=np.float32)\nclass VggFace(nn.Module):\n    """VGG-Face (Parkhi 2015) with pytorch-mcn `vgg_face_dag` layer names. fc7 after ReLU, BGR minus mean\n    reproduces the BTC face CSV (cos 1.0000 checked on dev)."""\n\n    def __init__(self):\n        super().__init__()\n        cfg = [(3, 64), (64, 64), \'P\', (64, 128), (128, 128), \'P\', (128, 256), (256, 256), (256, 256), \'P\',\n               (256, 512), (512, 512), (512, 512), \'P\', (512, 512), (512, 512), (512, 512), \'P\']\n        block, idx, self.order = 1, 1, []\n        for c in cfg:\n            if c == \'P\':\n                self.order.append(None); block += 1; idx = 1\n            else:\n                name = f\'conv{block}_{idx}\'\n                setattr(self, name, nn.Conv2d(c[0], c[1], 3, 1, 1)); self.order.append(name); idx += 1\n        self.fc6, self.fc7, self.fc8 = nn.Linear(25088, 4096), nn.Linear(4096, 4096), nn.Linear(4096, 2622)\n\n    def forward(self, x):\n        for name in self.order:\n            x = F.max_pool2d(x, 2, 2) if name is None else torch.relu(getattr(self, name)(x))\n        return torch.relu(self.fc7(torch.relu(self.fc6(x.flatten(1)))))\ndef vgg_input(img):\n    a = np.asarray(img.convert(\'RGB\').resize((224, 224), Image.BILINEAR), dtype=np.float32) - MEAN_RGB\n    return a[:, :, ::-1].transpose(2, 0, 1).copy()\ndef media_paths(list_path, rel_paths):\n    base = Path(list_path).parent\n    return [str(base / r) for r in rel_paths]\ndef dev_list_path(root, protocol, lang):\n    hits = set(p.resolve() for p in list(Path(root).rglob(f\'{protocol}_{lang}_test.txt\')) +\n               [p for p in Path(root).rglob(f\'{lang}_test.txt\') if protocol in p.parts] if p.is_file())\n    if len(hits) != 1:\n        raise ValueError(f\'{protocol}/{lang}: expected one trial list, found {sorted(map(str, hits))}\')\n    return next(iter(hits))\ndef fill_nonfinite(X, fallback_row):\n    """N2: replace non-finite rows by a constant fitted on train; return count."""\n    bad = ~np.isfinite(X).all(1)\n    X = X.copy(); X[bad] = fallback_row\n    return X, int(bad.sum())\ndef load_dev_lenient(root, protocol, lang, dims, strict_count, fallback_face, fallback_voice):\n    """N2: like load_dev, but non-finite rows are replaced by train-fitted constants and counted."""\n    paths = {}\n    for kind, flat, nested in ((\'face\', f\'{protocol}_{lang}_faces.csv\', f\'{lang}_test_faces.csv\'),\n                               (\'voice\', f\'{protocol}_{lang}_voices.csv\', f\'{lang}_test_voices.csv\')):\n        hits = set(p.resolve() for p in list(Path(root).rglob(flat)) + [p for p in Path(root).rglob(nested) if protocol in p.parts] if p.is_file())\n        if len(hits) != 1: raise ValueError(f\'{protocol}/{lang}/{kind}: expected 1 file, found {sorted(map(str, hits))}\')\n        paths[kind] = next(iter(hits))\n    f = pd.read_csv(paths[\'face\'], header=None).to_numpy(np.float32)\n    v = pd.read_csv(paths[\'voice\'], header=None).to_numpy(np.float32)\n    t = table(dev_list_path(root, protocol, lang), [\'pair_id\', \'voice\', \'face\'])\n    if not len(f) == len(v) == len(t): raise ValueError(f\'{protocol}/{lang}: counts differ\')\n    if (f.shape[1], v.shape[1]) != tuple(dims): raise ValueError(f\'{protocol}/{lang}: dims {f.shape[1], v.shape[1]}\')\n    if strict_count and len(t) != EXPECTED_ROWS[(protocol, lang)]: raise ValueError(f\'{protocol}/{lang}: row count\')\n    f, nf = fill_nonfinite(f, fallback_face); v, nv = fill_nonfinite(v, fallback_voice)\n    return f, v, t, dict(nonfinite_face_rows=nf, nonfinite_voice_rows=nv)\nfrom collections import OrderedDict\nclass TSTP(nn.Module):\n    """\n    Temporal statistics pooling, concatenate mean and std, which is used in\n    x-vector\n    Comment: simple concatenation can not make full use of both statistics\n    """\n\n    def __init__(self, in_dim=0, **kwargs):\n        super(TSTP, self).__init__()\n        self.in_dim = in_dim\n\n    def forward(self, x):\n        # The last dimension is the temporal axis\n        pooling_mean = x.mean(dim=-1)\n        pooling_std = torch.sqrt(torch.var(x, dim=-1) + 1e-7)\n        pooling_mean = pooling_mean.flatten(start_dim=1)\n        pooling_std = pooling_std.flatten(start_dim=1)\n        stats = torch.cat((pooling_mean, pooling_std), 1)\n        return stats\n\n    def get_out_dim(self):\n        self.out_dim = self.in_dim * 2\n        return self.out_dim\nclass ASTP(nn.Module):\n    """ Attentive statistics pooling: Channel- and context-dependent\n        statistics pooling, first used in ECAPA_TDNN.\n    """\n\n    def __init__(self,\n                 in_dim,\n                 bottleneck_dim=128,\n                 global_context_att=False,\n                 **kwargs):\n        super(ASTP, self).__init__()\n        self.in_dim = in_dim\n        self.global_context_att = global_context_att\n\n        # Use Conv1d with stride == 1 rather than Linear, then we don\'t\n        # need to transpose inputs.\n        if global_context_att:\n            self.linear1 = nn.Conv1d(\n                in_dim * 3, bottleneck_dim,\n                kernel_size=1)  # equals W and b in the paper\n        else:\n            self.linear1 = nn.Conv1d(\n                in_dim, bottleneck_dim,\n                kernel_size=1)  # equals W and b in the paper\n        self.linear2 = nn.Conv1d(bottleneck_dim, in_dim,\n                                 kernel_size=1)  # equals V and k in the paper\n\n    def forward(self, x):\n        """\n        x: a 3-dimensional tensor in tdnn-based architecture (B,F,T)\n            or a 4-dimensional tensor in resnet architecture (B,C,F,T)\n            0-dim: batch-dimension, last-dim: time-dimension (frame-dimension)\n        """\n        if len(x.shape) == 4:\n            x = x.reshape(x.shape[0], x.shape[1] * x.shape[2], x.shape[3])\n        assert len(x.shape) == 3\n\n        if self.global_context_att:\n            context_mean = torch.mean(x, dim=-1, keepdim=True).expand_as(x)\n            context_std = torch.sqrt(\n                torch.var(x, dim=-1, keepdim=True) + 1e-7).expand_as(x)\n            x_in = torch.cat((x, context_mean, context_std), dim=1)\n        else:\n            x_in = x\n\n        # DON\'T use ReLU here! ReLU may be hard to converge.\n        alpha = torch.tanh(\n            self.linear1(x_in))  # alpha = F.relu(self.linear1(x_in))\n        alpha = torch.softmax(self.linear2(alpha), dim=2)\n        mean = torch.sum(alpha * x, dim=2)\n        var = torch.sum(alpha * (x**2), dim=2) - mean**2\n        std = torch.sqrt(var.clamp(min=1e-7))\n        return torch.cat([mean, std], dim=1)\n\n    def get_out_dim(self):\n        self.out_dim = 2 * self.in_dim\n        return self.out_dim\nclass ASP(nn.Module):\n    """Attentive Statistics Pooling (compatible with WeSpeaker and\n    W2V-BERT/WavLM)."""\n\n    def __init__(\n        self,\n        in_planes=None,\n        acoustic_dim=None,\n        input_dim=None,\n        hidden_dim=None,\n    ):\n        super().__init__()\n\n        if input_dim is not None:\n            # W2V / Transformer-style input\n            self.feature_dim = input_dim\n        elif in_planes is not None and acoustic_dim is not None:\n            # WeSpeaker-style input\n            outmap_size = int(acoustic_dim / 8)\n            self.feature_dim = in_planes * 8 * outmap_size\n        else:\n            raise ValueError(\n                "Specify either (in_planes, acoustic_dim) or "\n                "(input_dim, hidden_dim)."\n            )\n\n        self.out_dim = self.feature_dim * 2\n        hidden_dim = hidden_dim or 128\n\n        self.attention = nn.Sequential(\n            nn.Conv1d(self.feature_dim, hidden_dim, kernel_size=1),\n            nn.ReLU(inplace=True),\n            nn.BatchNorm1d(hidden_dim),\n            nn.Conv1d(hidden_dim, self.feature_dim, kernel_size=1),\n            nn.Softmax(dim=2),\n        )\n\n    def forward(self, x: torch.Tensor):\n        # W2V: [B, T, D]\n        # WeSpeaker: [B, C, F, T]\n        if x.dim() == 4:\n            x = x.reshape(x.size(0), -1, x.size(-1))\n        elif x.dim() == 3 and x.shape[1] != self.feature_dim:\n            x = x.transpose(1, 2)\n\n        w = self.attention(x)\n        mu = torch.sum(x * w, dim=2)\n        sg = torch.sqrt(\n            (\n                torch.sum((x**2) * w, dim=2)\n                - mu**2\n            ).clamp(min=1e-5)\n        )\n        return torch.cat([mu, sg], dim=1)\nPOOLING = {\'TSTP\': TSTP, \'ASTP\': ASTP, \'ASP\': ASP}\nclass SimAMBasicBlock(nn.Module):\n    expansion = 1\n\n    def __init__(\n        self, ConvLayer, NormLayer, in_planes, planes, stride=1, block_id=1\n    ):\n        super(SimAMBasicBlock, self).__init__()\n        self.conv1 = ConvLayer(\n            in_planes,\n            planes,\n            kernel_size=3,\n            stride=stride,\n            padding=1,\n            bias=False,\n        )\n        self.bn1 = NormLayer(planes)\n        self.conv2 = ConvLayer(\n            planes, planes, kernel_size=3, stride=1, padding=1, bias=False\n        )\n        self.bn2 = NormLayer(planes)\n        self.relu = nn.ReLU(inplace=True)\n        self.sigmoid = nn.Sigmoid()\n\n        self.downsample = nn.Sequential()\n        if stride != 1 or in_planes != self.expansion * planes:\n            self.downsample = nn.Sequential(\n                ConvLayer(\n                    in_planes,\n                    self.expansion * planes,\n                    kernel_size=1,\n                    stride=stride,\n                    bias=False,\n                ),\n                NormLayer(self.expansion * planes),\n            )\n\n    def forward(self, x):\n        out = self.relu(self.bn1(self.conv1(x)))\n        out = self.bn2(self.conv2(out))\n        out = self.SimAM(out)\n        out += self.downsample(x)\n        out = self.relu(out)\n        return out\n\n    def SimAM(self, X, lambda_p=1e-4):\n        n = X.shape[2] * X.shape[3] - 1\n        d = (X - X.mean(dim=[2, 3], keepdim=True)).pow(2)\n        v = d.sum(dim=[2, 3], keepdim=True) / n\n        E_inv = d / (4 * (v + lambda_p)) + 0.5\n        return X * self.sigmoid(E_inv)\nclass SamResNet(nn.Module):\n    def __init__(\n        self, in_planes, block, num_blocks, in_ch=1, **kwargs\n    ):\n        super(SamResNet, self).__init__()\n        self.in_planes = in_planes\n        self.NormLayer = nn.BatchNorm2d\n        self.ConvLayer = nn.Conv2d\n\n        self.conv1 = self.ConvLayer(\n            in_ch, in_planes, kernel_size=3, stride=1, padding=1, bias=False\n        )\n        self.bn1 = self.NormLayer(in_planes)\n        self.relu = nn.ReLU(inplace=True)\n        self.layer1 = self._make_layer(\n            block, in_planes, num_blocks[0], stride=1, block_id=1\n        )\n        self.layer2 = self._make_layer(\n            block, in_planes * 2, num_blocks[1], stride=2, block_id=2\n        )\n        self.layer3 = self._make_layer(\n            block, in_planes * 4, num_blocks[2], stride=2, block_id=3\n        )\n        self.layer4 = self._make_layer(\n            block, in_planes * 8, num_blocks[3], stride=2, block_id=4\n        )\n\n    def _make_layer(self, block, planes, num_blocks, stride, block_id=1):\n        strides = [stride] + [1] * (num_blocks - 1)\n        layers = []\n        for stride in strides:\n            layers.append(\n                block(\n                    self.ConvLayer,\n                    self.NormLayer,\n                    self.in_planes,\n                    planes,\n                    stride,\n                    block_id,\n                )\n            )\n            self.in_planes = planes * block.expansion\n        return nn.Sequential(*layers)\n\n    def forward(self, x):\n        x = self.relu(self.bn1(self.conv1(x)))\n        x = self.layer1(x)\n        x = self.layer2(x)\n        x = self.layer3(x)\n        x = self.layer4(x)\n        return x\ndef SimAM_ResNet100(in_planes):\n    return SamResNet(in_planes, SimAMBasicBlock, [6, 16, 24, 3])\nclass SimAM_ResNet100_ASP(nn.Module):\n    def __init__(self, in_planes=64, embed_dim=256, acoustic_dim=80, dropout=0):\n        super(SimAM_ResNet100_ASP, self).__init__()\n        self.front = SimAM_ResNet100(in_planes)\n        self.pooling = ASP(in_planes, acoustic_dim)\n        self.bottleneck = nn.Linear(self.pooling.out_dim, embed_dim)\n        self.drop = nn.Dropout(dropout) if dropout else None\n\n    def forward(self, x):\n        x = x.permute(0, 2, 1)\n        x = self.front(x.unsqueeze(dim=1))\n        x = self.pooling(x)\n        if self.drop:\n            x = self.drop(x)\n        x = self.bottleneck(x)\n        return x\nclass Bottleneck(nn.Module):\n    expansion = 4\n\n    def __init__(self, in_planes, planes, stride=1):\n        super(Bottleneck, self).__init__()\n        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)\n        self.bn1 = nn.BatchNorm2d(planes)\n        self.conv2 = nn.Conv2d(planes,\n                               planes,\n                               kernel_size=3,\n                               stride=stride,\n                               padding=1,\n                               bias=False)\n        self.bn2 = nn.BatchNorm2d(planes)\n        self.conv3 = nn.Conv2d(planes,\n                               self.expansion * planes,\n                               kernel_size=1,\n                               bias=False)\n        self.bn3 = nn.BatchNorm2d(self.expansion * planes)\n\n        self.shortcut = nn.Sequential()\n        if stride != 1 or in_planes != self.expansion * planes:\n            self.shortcut = nn.Sequential(\n                nn.Conv2d(in_planes,\n                          self.expansion * planes,\n                          kernel_size=1,\n                          stride=stride,\n                          bias=False), nn.BatchNorm2d(self.expansion * planes))\n\n    def forward(self, x):\n        out = F.relu(self.bn1(self.conv1(x)))\n        out = F.relu(self.bn2(self.conv2(out)))\n        out = self.bn3(self.conv3(out))\n        out += self.shortcut(x)\n        out = F.relu(out)\n        return out\nclass ResNet(nn.Module):\n\n    def __init__(self,\n                 block,\n                 num_blocks,\n                 m_channels=32,\n                 feat_dim=40,\n                 embed_dim=128,\n                 pooling_func=\'TSTP\',\n                 two_emb_layer=False):\n        super(ResNet, self).__init__()\n        self.in_planes = m_channels\n        self.feat_dim = feat_dim\n        self.embed_dim = embed_dim\n        self.stats_dim = int(feat_dim / 8) * m_channels * 8\n        self.two_emb_layer = two_emb_layer\n\n        self.conv1 = nn.Conv2d(1,\n                               m_channels,\n                               kernel_size=3,\n                               stride=1,\n                               padding=1,\n                               bias=False)\n        self.bn1 = nn.BatchNorm2d(m_channels)\n        self.layer1 = self._make_layer(block,\n                                       m_channels,\n                                       num_blocks[0],\n                                       stride=1)\n        self.layer2 = self._make_layer(block,\n                                       m_channels * 2,\n                                       num_blocks[1],\n                                       stride=2)\n        self.layer3 = self._make_layer(block,\n                                       m_channels * 4,\n                                       num_blocks[2],\n                                       stride=2)\n        self.layer4 = self._make_layer(block,\n                                       m_channels * 8,\n                                       num_blocks[3],\n                                       stride=2)\n\n        self.pool = POOLING[pooling_func](in_dim=self.stats_dim *\n                                          block.expansion)\n        self.pool_out_dim = self.pool.get_out_dim()\n        self.seg_1 = nn.Linear(self.pool_out_dim, embed_dim)\n        if self.two_emb_layer:\n            self.seg_bn_1 = nn.BatchNorm1d(embed_dim, affine=False)\n            self.seg_2 = nn.Linear(embed_dim, embed_dim)\n        else:\n            self.seg_bn_1 = nn.Identity()\n            self.seg_2 = nn.Identity()\n\n    def _make_layer(self, block, planes, num_blocks, stride):\n        strides = [stride] + [1] * (num_blocks - 1)\n        layers = []\n        for stride in strides:\n            layers.append(block(self.in_planes, planes, stride))\n            self.in_planes = planes * block.expansion\n        return nn.Sequential(*layers)\n\n    def _get_frame_level_feat(self, x):\n        # for inner class usage\n        x = x.permute(0, 2, 1)  # (B,T,F) => (B,F,T)\n\n        x = x.unsqueeze_(1)\n        out = F.relu(self.bn1(self.conv1(x)))\n        out = self.layer1(out)\n        out = self.layer2(out)\n        out = self.layer3(out)\n        out = self.layer4(out)\n\n        return out\n\n    def get_frame_level_feat(self, x):\n        # for outer interface\n        out = self._get_frame_level_feat(x)\n        out = out.transpose(1, 3)\n        out = torch.flatten(out, 2, -1)\n\n        return out  # (B, T, D)\n\n    def forward(self, x):\n        out = self._get_frame_level_feat(x)\n\n        stats = self.pool(out)\n\n        embed_a = self.seg_1(stats)\n        if self.two_emb_layer:\n            out = F.relu(embed_a)\n            out = self.seg_bn_1(out)\n            embed_b = self.seg_2(out)\n            return embed_a, embed_b\n        else:\n            return torch.tensor(0.0), embed_a\ndef ResNet221(feat_dim, embed_dim, pooling_func=\'TSTP\', two_emb_layer=False):\n    return ResNet(Bottleneck, [6, 16, 48, 3],\n                  feat_dim=feat_dim,\n                  embed_dim=embed_dim,\n                  pooling_func=pooling_func,\n                  two_emb_layer=two_emb_layer)\ndef ResNet293(feat_dim, embed_dim, pooling_func=\'TSTP\', two_emb_layer=False):\n    return ResNet(Bottleneck, [10, 20, 64, 3],\n                  feat_dim=feat_dim,\n                  embed_dim=embed_dim,\n                  pooling_func=pooling_func,\n                  two_emb_layer=two_emb_layer)\nclass Res2Conv1dReluBn(nn.Module):\n    """\n    in_channels == out_channels == channels\n    """\n\n    def __init__(self,\n                 channels,\n                 kernel_size=1,\n                 stride=1,\n                 padding=0,\n                 dilation=1,\n                 bias=True,\n                 scale=4):\n        super().__init__()\n        assert channels % scale == 0, "{} % {} != 0".format(channels, scale)\n        self.scale = scale\n        self.width = channels // scale\n        self.nums = scale if scale == 1 else scale - 1\n\n        self.convs = []\n        self.bns = []\n        for i in range(self.nums):\n            self.convs.append(\n                nn.Conv1d(self.width,\n                          self.width,\n                          kernel_size,\n                          stride,\n                          padding,\n                          dilation,\n                          bias=bias))\n            self.bns.append(nn.BatchNorm1d(self.width))\n        self.convs = nn.ModuleList(self.convs)\n        self.bns = nn.ModuleList(self.bns)\n\n    def forward(self, x):\n        out = []\n        spx = torch.split(x, self.width, 1)\n        sp = spx[0]\n        for i, (conv, bn) in enumerate(zip(self.convs, self.bns)):\n            # Order: conv -> relu -> bn\n            if i >= 1:\n                sp = sp + spx[i]\n            sp = conv(sp)\n            sp = bn(F.relu(sp))\n            out.append(sp)\n        if self.scale != 1:\n            out.append(spx[self.nums])\n        out = torch.cat(out, dim=1)\n\n        return out\nclass Conv1dReluBn(nn.Module):\n\n    def __init__(self,\n                 in_channels,\n                 out_channels,\n                 kernel_size=1,\n                 stride=1,\n                 padding=0,\n                 dilation=1,\n                 bias=True):\n        super().__init__()\n        self.conv = nn.Conv1d(in_channels,\n                              out_channels,\n                              kernel_size,\n                              stride,\n                              padding,\n                              dilation,\n                              bias=bias)\n        self.bn = nn.BatchNorm1d(out_channels)\n\n    def forward(self, x):\n        return self.bn(F.relu(self.conv(x)))\nclass SE_Connect(nn.Module):\n\n    def __init__(self, channels, se_bottleneck_dim=128):\n        super().__init__()\n        self.linear1 = nn.Linear(channels, se_bottleneck_dim)\n        self.linear2 = nn.Linear(se_bottleneck_dim, channels)\n\n    def forward(self, x):\n        out = x.mean(dim=2)\n        out = F.relu(self.linear1(out))\n        out = torch.sigmoid(self.linear2(out))\n        out = x * out.unsqueeze(2)\n\n        return out\nclass SE_Res2Block(nn.Module):\n\n    def __init__(self, channels, kernel_size, stride, padding, dilation,\n                 scale):\n        super().__init__()\n        self.se_res2block = nn.Sequential(\n            Conv1dReluBn(channels,\n                         channels,\n                         kernel_size=1,\n                         stride=1,\n                         padding=0),\n            Res2Conv1dReluBn(channels,\n                             kernel_size,\n                             stride,\n                             padding,\n                             dilation,\n                             scale=scale),\n            Conv1dReluBn(channels,\n                         channels,\n                         kernel_size=1,\n                         stride=1,\n                         padding=0), SE_Connect(channels))\n\n    def forward(self, x):\n        return x + self.se_res2block(x)\nclass ECAPA_TDNN(nn.Module):\n\n    def __init__(self,\n                 channels=512,\n                 feat_dim=80,\n                 embed_dim=192,\n                 pooling_func=\'ASTP\',\n                 global_context_att=False,\n                 emb_bn=False):\n        super().__init__()\n\n        self.layer1 = Conv1dReluBn(feat_dim,\n                                   channels,\n                                   kernel_size=5,\n                                   padding=2)\n        self.layer2 = SE_Res2Block(channels,\n                                   kernel_size=3,\n                                   stride=1,\n                                   padding=2,\n                                   dilation=2,\n                                   scale=8)\n        self.layer3 = SE_Res2Block(channels,\n                                   kernel_size=3,\n                                   stride=1,\n                                   padding=3,\n                                   dilation=3,\n                                   scale=8)\n        self.layer4 = SE_Res2Block(channels,\n                                   kernel_size=3,\n                                   stride=1,\n                                   padding=4,\n                                   dilation=4,\n                                   scale=8)\n\n        cat_channels = channels * 3\n        out_channels = 512 * 3\n        self.conv = nn.Conv1d(cat_channels, out_channels, kernel_size=1)\n        self.pool = POOLING[pooling_func](\n            in_dim=out_channels, global_context_att=global_context_att)\n        self.pool_out_dim = self.pool.get_out_dim()\n        self.bn = nn.BatchNorm1d(self.pool_out_dim)\n        self.linear = nn.Linear(self.pool_out_dim, embed_dim)\n        self.emb_bn = emb_bn\n        if emb_bn:  # better in SSL for SV\n            self.bn2 = nn.BatchNorm1d(embed_dim)\n        else:\n            self.bn2 = nn.Identity()\n\n    def _get_frame_level_feat(self, x):\n        # for inner class usage\n        x = x.permute(0, 2, 1)  # (B,T,F) -> (B,F,T)\n\n        out1 = self.layer1(x)\n        out2 = self.layer2(out1)\n        out3 = self.layer3(out2)\n        out4 = self.layer4(out3)\n\n        out = torch.cat([out2, out3, out4], dim=1)\n        out = self.conv(out)\n\n        return out, out4\n\n    def get_frame_level_feat(self, x):\n        # for outer interface\n        out = self._get_frame_level_feat(x)[0].permute(0, 2, 1)\n        return out  # (B, T, D)\n\n    def forward(self, x):\n        out, out4 = self._get_frame_level_feat(x)\n        out = F.relu(out)\n        out = self.bn(self.pool(out))\n        out = self.linear(out)\n        if self.emb_bn:\n            out = self.bn2(out)\n        return out4, out\ndef ECAPA_TDNN_GLOB_c1024(feat_dim,\n                          embed_dim,\n                          pooling_func=\'ASTP\',\n                          emb_bn=False):\n    return ECAPA_TDNN(channels=1024,\n                      feat_dim=feat_dim,\n                      embed_dim=embed_dim,\n                      pooling_func=pooling_func,\n                      global_context_att=True,\n                      emb_bn=emb_bn)\ndef get_nonlinear(config_str, channels):\n    nonlinear = nn.Sequential()\n    for name in config_str.split(\'-\'):\n        if name == \'relu\':\n            nonlinear.add_module(\'relu\', nn.ReLU(inplace=True))\n        elif name == \'prelu\':\n            nonlinear.add_module(\'prelu\', nn.PReLU(channels))\n        elif name == \'batchnorm\':\n            nonlinear.add_module(\'batchnorm\', nn.BatchNorm1d(channels))\n        elif name == \'batchnorm_\':\n            nonlinear.add_module(\'batchnorm\',\n                                 nn.BatchNorm1d(channels, affine=False))\n        else:\n            raise ValueError(\'Unexpected module ({}).\'.format(name))\n    return nonlinear\nclass TDNNLayer(nn.Module):\n\n    def __init__(self,\n                 in_channels,\n                 out_channels,\n                 kernel_size,\n                 stride=1,\n                 padding=0,\n                 dilation=1,\n                 bias=False,\n                 config_str=\'batchnorm-relu\'):\n        super(TDNNLayer, self).__init__()\n        if padding < 0:\n            assert kernel_size % 2 == 1, \'Expect equal paddings, \\\n                    but got even kernel size ({})\'.format(kernel_size)\n            padding = (kernel_size - 1) // 2 * dilation\n        self.linear = nn.Conv1d(in_channels,\n                                out_channels,\n                                kernel_size,\n                                stride=stride,\n                                padding=padding,\n                                dilation=dilation,\n                                bias=bias)\n        self.nonlinear = get_nonlinear(config_str, out_channels)\n\n    def forward(self, x):\n        x = self.linear(x)\n        x = self.nonlinear(x)\n        return x\nclass CAMLayer(nn.Module):\n\n    def __init__(self,\n                 bn_channels,\n                 out_channels,\n                 kernel_size,\n                 stride,\n                 padding,\n                 dilation,\n                 bias,\n                 reduction=2):\n        super(CAMLayer, self).__init__()\n        self.linear_local = nn.Conv1d(bn_channels,\n                                      out_channels,\n                                      kernel_size,\n                                      stride=stride,\n                                      padding=padding,\n                                      dilation=dilation,\n                                      bias=bias)\n        self.linear1 = nn.Conv1d(bn_channels, bn_channels // reduction, 1)\n        self.relu = nn.ReLU(inplace=True)\n        self.linear2 = nn.Conv1d(bn_channels // reduction, out_channels, 1)\n        self.sigmoid = nn.Sigmoid()\n\n    def forward(self, x):\n        y = self.linear_local(x)\n        context = x.mean(-1, keepdim=True) + self.seg_pooling(x)\n        context = self.relu(self.linear1(context))\n        m = self.sigmoid(self.linear2(context))\n        return y * m\n\n    def seg_pooling(self, x, seg_len: int = 100, stype: str = \'avg\'):\n        if stype == \'avg\':\n            seg = F.avg_pool1d(x,\n                               kernel_size=seg_len,\n                               stride=seg_len,\n                               ceil_mode=True)\n        elif stype == \'max\':\n            seg = F.max_pool1d(x,\n                               kernel_size=seg_len,\n                               stride=seg_len,\n                               ceil_mode=True)\n        else:\n            raise ValueError(\'Wrong segment pooling type.\')\n        shape = seg.shape\n        seg = seg.unsqueeze(-1).expand(shape[0], shape[1], shape[2],\n                                       seg_len).reshape(\n                                           shape[0], shape[1], -1)\n        seg = seg[..., :x.shape[-1]]\n        return seg\nclass CAMDenseTDNNLayer(nn.Module):\n\n    def __init__(self,\n                 in_channels,\n                 out_channels,\n                 bn_channels,\n                 kernel_size,\n                 stride=1,\n                 dilation=1,\n                 bias=False,\n                 config_str=\'batchnorm-relu\'):\n        super(CAMDenseTDNNLayer, self).__init__()\n        assert kernel_size % 2 == 1, \'Expect equal paddings, \\\n                but got even kernel size ({})\'.format(kernel_size)\n        padding = (kernel_size - 1) // 2 * dilation\n        self.nonlinear1 = get_nonlinear(config_str, in_channels)\n        self.linear1 = nn.Conv1d(in_channels, bn_channels, 1, bias=False)\n        self.nonlinear2 = get_nonlinear(config_str, bn_channels)\n        self.cam_layer = CAMLayer(bn_channels,\n                                  out_channels,\n                                  kernel_size,\n                                  stride=stride,\n                                  padding=padding,\n                                  dilation=dilation,\n                                  bias=bias)\n\n    def bn_function(self, x):\n        return self.linear1(self.nonlinear1(x))\n\n    def forward(self, x):\n        x = self.bn_function(x)\n        x = self.cam_layer(self.nonlinear2(x))\n        return x\nclass CAMDenseTDNNBlock(nn.ModuleList):\n\n    def __init__(self,\n                 num_layers,\n                 in_channels,\n                 out_channels,\n                 bn_channels,\n                 kernel_size,\n                 stride=1,\n                 dilation=1,\n                 bias=False,\n                 config_str=\'batchnorm-relu\'):\n        super(CAMDenseTDNNBlock, self).__init__()\n        for i in range(num_layers):\n            layer = CAMDenseTDNNLayer(in_channels=in_channels +\n                                      i * out_channels,\n                                      out_channels=out_channels,\n                                      bn_channels=bn_channels,\n                                      kernel_size=kernel_size,\n                                      stride=stride,\n                                      dilation=dilation,\n                                      bias=bias,\n                                      config_str=config_str)\n            self.add_module(\'tdnnd%d\' % (i + 1), layer)\n\n    def forward(self, x):\n        for layer in self:\n            x = torch.cat([x, layer(x)], dim=1)\n        return x\nclass TransitLayer(nn.Module):\n\n    def __init__(self,\n                 in_channels,\n                 out_channels,\n                 bias=True,\n                 config_str=\'batchnorm-relu\'):\n        super(TransitLayer, self).__init__()\n        self.nonlinear = get_nonlinear(config_str, in_channels)\n        self.linear = nn.Conv1d(in_channels, out_channels, 1, bias=bias)\n\n    def forward(self, x):\n        x = self.nonlinear(x)\n        x = self.linear(x)\n        return x\nclass DenseLayer(nn.Module):\n\n    def __init__(self,\n                 in_channels,\n                 out_channels,\n                 bias=False,\n                 config_str=\'batchnorm-relu\'):\n        super(DenseLayer, self).__init__()\n        self.linear = nn.Conv1d(in_channels, out_channels, 1, bias=bias)\n        self.nonlinear = get_nonlinear(config_str, out_channels)\n\n    def forward(self, x):\n        if len(x.shape) == 2:\n            x = self.linear(x.unsqueeze(dim=-1)).squeeze(dim=-1)\n        else:\n            x = self.linear(x)\n        x = self.nonlinear(x)\n        return x\nclass BasicResBlock(nn.Module):\n    expansion = 1\n\n    def __init__(self, in_planes, planes, stride=1):\n        super(BasicResBlock, self).__init__()\n        self.conv1 = nn.Conv2d(in_planes,\n                               planes,\n                               kernel_size=3,\n                               stride=(stride, 1),\n                               padding=1,\n                               bias=False)\n        self.bn1 = nn.BatchNorm2d(planes)\n        self.conv2 = nn.Conv2d(planes,\n                               planes,\n                               kernel_size=3,\n                               stride=1,\n                               padding=1,\n                               bias=False)\n        self.bn2 = nn.BatchNorm2d(planes)\n\n        self.shortcut = nn.Sequential()\n        if stride != 1 or in_planes != self.expansion * planes:\n            self.shortcut = nn.Sequential(\n                nn.Conv2d(in_planes,\n                          self.expansion * planes,\n                          kernel_size=1,\n                          stride=(stride, 1),\n                          bias=False), nn.BatchNorm2d(self.expansion * planes))\n\n    def forward(self, x):\n        out = F.relu(self.bn1(self.conv1(x)))\n        out = self.bn2(self.conv2(out))\n        out += self.shortcut(x)\n        out = F.relu(out)\n        return out\nclass FCM(nn.Module):\n\n    def __init__(self, block, num_blocks, m_channels=32, feat_dim=80):\n        super(FCM, self).__init__()\n        self.in_planes = m_channels\n        self.conv1 = nn.Conv2d(1,\n                               m_channels,\n                               kernel_size=3,\n                               stride=1,\n                               padding=1,\n                               bias=False)\n        self.bn1 = nn.BatchNorm2d(m_channels)\n\n        self.layer1 = self._make_layer(block,\n                                       m_channels,\n                                       num_blocks[0],\n                                       stride=2)\n        self.layer2 = self._make_layer(block,\n                                       m_channels,\n                                       num_blocks[1],\n                                       stride=2)\n\n        self.conv2 = nn.Conv2d(m_channels,\n                               m_channels,\n                               kernel_size=3,\n                               stride=(2, 1),\n                               padding=1,\n                               bias=False)\n        self.bn2 = nn.BatchNorm2d(m_channels)\n        self.out_channels = m_channels * (feat_dim // 8)\n\n    def _make_layer(self, block, planes, num_blocks, stride):\n        strides = [stride] + [1] * (num_blocks - 1)\n        layers = []\n        for stride in strides:\n            layers.append(block(self.in_planes, planes, stride))\n            self.in_planes = planes * block.expansion\n        return nn.Sequential(*layers)\n\n    def forward(self, x):\n        x = x.unsqueeze(1)\n        out = F.relu(self.bn1(self.conv1(x)))\n        out = self.layer1(out)\n        out = self.layer2(out)\n        out = F.relu(self.bn2(self.conv2(out)))\n\n        shape = out.shape\n        out = out.reshape(shape[0], shape[1] * shape[2], shape[3])\n        return out\nclass CAMPPlus(nn.Module):\n\n    def __init__(self,\n                 feat_dim=80,\n                 embed_dim=512,\n                 pooling_func=\'TSTP\',\n                 growth_rate=32,\n                 bn_size=4,\n                 init_channels=128,\n                 config_str=\'batchnorm-relu\'):\n        super(CAMPPlus, self).__init__()\n\n        self.head = FCM(block=BasicResBlock,\n                        num_blocks=[2, 2],\n                        feat_dim=feat_dim)\n        channels = self.head.out_channels\n\n        self.xvector = nn.Sequential(\n            OrderedDict([\n                (\'tdnn\',\n                 TDNNLayer(channels,\n                           init_channels,\n                           5,\n                           stride=2,\n                           dilation=1,\n                           padding=-1,\n                           config_str=config_str)),\n            ]))\n        channels = init_channels\n        for i, (num_layers, kernel_size,\n                dilation) in enumerate(zip((12, 24, 16), (3, 3, 3),\n                                           (1, 2, 2))):\n            block = CAMDenseTDNNBlock(num_layers=num_layers,\n                                      in_channels=channels,\n                                      out_channels=growth_rate,\n                                      bn_channels=bn_size * growth_rate,\n                                      kernel_size=kernel_size,\n                                      dilation=dilation,\n                                      config_str=config_str)\n            self.xvector.add_module(\'block%d\' % (i + 1), block)\n            channels = channels + num_layers * growth_rate\n            self.xvector.add_module(\n                \'transit%d\' % (i + 1),\n                TransitLayer(channels,\n                             channels // 2,\n                             bias=False,\n                             config_str=config_str))\n            channels //= 2\n\n        self.xvector.add_module(\'out_nonlinear\',\n                                get_nonlinear(config_str, channels))\n\n        self.pool = POOLING[pooling_func](in_dim=channels)\n        self.pool_out_dim = self.pool.get_out_dim()\n        self.xvector.add_module(\'stats\', self.pool)\n        self.xvector.add_module(\n            \'dense\',\n            DenseLayer(self.pool_out_dim, embed_dim, config_str=\'batchnorm_\'))\n\n        for m in self.modules():\n            if isinstance(m, (nn.Conv1d, nn.Linear)):\n                nn.init.kaiming_normal_(m.weight.data)\n                if m.bias is not None:\n                    nn.init.zeros_(m.bias)\n\n    def get_frame_level_feat(self, x):\n        # for outer interface\n        x = x.permute(0, 2, 1)  # (B,T,F) => (B,F,T)\n        x = self.head(x)\n        for layer in self.xvector[:-2]:\n            x = layer(x)\n\n        out = x.permute(0, 2, 1)\n\n        return out  # (B, T, D)\n\n    def forward(self, x):\n        x = x.permute(0, 2, 1)  # (B,T,F) => (B,F,T)\n        x = self.head(x)\n        x = self.xvector(x)\n        return x\nVERSION = "RF-L0-004-v1"\ndef kaldi_fbank(wave):\n    """WeSpeaker front end: 80-dim Kaldi fbank on int16-scaled audio, per-utterance mean removal (this clip only)."""\n    import torchaudio.compliance.kaldi as kaldi\n    f = kaldi.fbank(torch.as_tensor(wave)[None] * 32768, num_mel_bins=80, frame_length=25, frame_shift=10,\n                    dither=0.0, sample_frequency=16000, window_type=\'hamming\')\n    return f - f.mean(0)\ndef load_wave(path, min_sec=1.0):\n    """Mono 16 kHz float32. N2: unreadable -> None; shorter than min_sec -> tiled (uses only this clip)."""\n    import soundfile as sf\n    try:\n        w, sr = sf.read(path, dtype=\'float32\', always_2d=True)\n    except Exception as e:\n        return None, f\'fallback_unreadable:{type(e).__name__}\'\n    w = w.mean(1)\n    if sr != 16000:\n        import torchaudio.functional as AF\n        w = AF.resample(torch.tensor(w), sr, 16000).numpy()\n    if len(w) == 0 or not np.isfinite(w).all() or np.abs(w).max() == 0:\n        return None, \'fallback_silent_or_empty\'\n    if len(w) < min_sec * 16000:\n        return np.tile(w, int(np.ceil(min_sec * 16000 / len(w))))[:int(min_sec * 16000)], \'tiled_short\'\n    return w, \'ok\'\nVERSION = "RF-L0-005-v1"\nENCODERS = {\n    \'resnet221_lm\': dict(make=lambda: ResNet221(feat_dim=80, embed_dim=256, pooling_func=\'TSTP\', two_emb_layer=False), dim=256),\n    \'resnet293_lm\': dict(make=lambda: ResNet293(feat_dim=80, embed_dim=256, pooling_func=\'TSTP\', two_emb_layer=False), dim=256),\n    \'ecapa1024_lm\': dict(make=lambda: ECAPA_TDNN_GLOB_c1024(feat_dim=80, embed_dim=192, pooling_func=\'ASTP\'), dim=192),\n    \'campplus_lm\': dict(make=lambda: CAMPPlus(feat_dim=80, embed_dim=512, pooling_func=\'TSTP\'), dim=512),\n}\nVERSION = "RF-L1-001-v1"\nIB_VIS_MEAN, IB_VIS_STD = (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)\ndef load_imagebind(ckpt):\n    """Vision + audio towers of ImageBind-huge in fp16, loaded without imagebind/__init__ (which needs pytorchvideo)\n    and without ever holding the 4.8 GB fp32 model in RAM."""\n    import sys as _sys, types as _types, importlib.util as _ilu\n    if \'imagebind\' not in _sys.modules:   # a second call must not run find_spec on the bare package (its __spec__ is None)\n        spec = _ilu.find_spec(\'imagebind\')\n        if spec is None:\n            raise ImportError(\'pip install --no-deps git+https://github.com/facebookresearch/ImageBind.git\')\n        pkg = _types.ModuleType(\'imagebind\'); pkg.__path__ = list(spec.submodule_search_locations)\n        _sys.modules[\'imagebind\'] = pkg\n    from imagebind.models import imagebind_model\n    from imagebind.models.imagebind_model import ModalityType\n    torch.set_default_dtype(torch.float16)\n    try:\n        m = imagebind_model.imagebind_huge(pretrained=False)\n    finally:\n        torch.set_default_dtype(torch.float32)\n    for part in (\'modality_preprocessors\', \'modality_trunks\', \'modality_heads\', \'modality_postprocessors\'):\n        for k in [k for k in getattr(m, part) if k not in (ModalityType.VISION, ModalityType.AUDIO)]:\n            del getattr(m, part)[k]\n    sd = torch.load(ckpt, map_location=\'cpu\', mmap=True)\n    sd = {k: v.to(torch.float16) for k, v in sd.items() if \'.vision\' in k or \'.audio\' in k}\n    res = m.load_state_dict(sd, strict=False)\n    if res.missing_keys:\n        raise ValueError(f\'ImageBind checkpoint mismatch: {res.missing_keys[:5]}\')\n    return m.eval().to(DEVICE), ModalityType\ndef ib_audio_clips(wave):\n    """ImageBind audio input: 2 s clips covering the whole file (>= 3), 128-bin fbank, 204 frames, AudioSet norm."""\n    import math\n    import torchaudio.compliance.kaldi as kaldi\n    w = torch.as_tensor(wave)\n    dur = len(w) / 16000\n    clips = []\n    for s in np.linspace(0, max(dur - 2, 0), max(3, math.ceil(dur / 2))):\n        c = w[int(s * 16000):int((s + 2) * 16000)]\n        c = c - c.mean()\n        f = kaldi.fbank(c[None], htk_compat=True, sample_frequency=16000, use_energy=False, window_type=\'hanning\',\n                        num_mel_bins=128, dither=0.0, frame_length=25, frame_shift=10).T\n        f = F.pad(f, (0, max(0, 204 - f.shape[1])))[:, :204]\n        clips.append(((f + 4.268) / 9.138)[None])\n    return torch.stack(clips)\nclass L1Extractor:\n    """Per-file embeddings: ImageBind face + voice, ResNet221 voice. Identical files are embedded once."""\n\n    def __init__(self, ib_ckpt, r221_ckpt):\n        from torchvision import transforms as T\n        self.ib, self.MT = load_imagebind(ib_ckpt)\n        self.vis = T.Compose([T.Resize(224, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224), T.ToTensor(),\n                              T.Normalize(IB_VIS_MEAN, IB_VIS_STD)])\n        r = ENCODERS[\'resnet221_lm\'][\'make\']()\n        sd = torch.load(r221_ckpt, map_location=\'cpu\'); sd = sd.get(\'model\', sd)\n        res = r.load_state_dict(sd, strict=False)\n        if res.missing_keys or set(res.unexpected_keys) - {\'projection.weight\'}:\n            raise ValueError(f\'ResNet221 checkpoint mismatch: {res}\')\n        self.r221 = r.to(DEVICE).eval()\n\n    @torch.inference_mode()\n    def faces(self, paths):\n        out, status, cache = [], [], {}\n        for p in paths:\n            try:\n                key = hashlib.md5(Path(p).read_bytes()).hexdigest()\n                if key not in cache:\n                    img = self.vis(Image.open(p).convert(\'RGB\'))[None].half().to(DEVICE)\n                    cache[key] = self.ib({self.MT.VISION: img})[self.MT.VISION].float().cpu().numpy()[0]\n                out.append(cache[key]); status.append(\'ok\')\n            except Exception as e:  # N2: unreadable image -> filled with a train constant later\n                out.append(np.full(1024, np.nan, np.float32)); status.append(f\'fallback_unreadable:{type(e).__name__}\')\n        return np.asarray(out, np.float32), status\n\n    @torch.inference_mode()\n    def voices(self, paths):\n        ib, r2, status, cache = [], [], [], {}\n        for i, p in enumerate(paths):\n            try:\n                key = hashlib.md5(Path(p).read_bytes()).hexdigest()\n            except Exception:\n                key = None\n            if key is not None and key in cache:\n                e, st = cache[key]\n            else:\n                w, st = load_wave(p)\n                if w is None:\n                    e = (np.full(1024, np.nan, np.float32), np.full(256, np.nan, np.float32))\n                else:\n                    a = ib_audio_clips(w)[None].half().to(DEVICE)\n                    e_ib = self.ib({self.MT.AUDIO: a})[self.MT.AUDIO].float().cpu().numpy()[0]\n                    o = self.r221(kaldi_fbank(w)[None].to(DEVICE))\n                    e = (e_ib, (o[-1] if isinstance(o, tuple) else o).squeeze(0).float().cpu().numpy())\n                if key is not None:\n                    cache[key] = (e, st)\n            ib.append(e[0]); r2.append(e[1]); status.append(st)\n            if (i + 1) % 1000 == 0:\n                print(f\'    {i + 1}/{len(paths)} voices ({len(cache)} unique)\', flush=True)\n        return np.asarray(ib, np.float32), np.asarray(r2, np.float32), status\ndef _unit(X):\n    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)\nVERSION = "RF-L1-002-v1"\nclass L2Extractor(L1Extractor):\n    """Adds the two BTC-space encoders to L1Extractor: VGG-Face fc7 (faces) and SpeechBrain ECAPA (voices)."""\n\n    def __init__(self, ib_ckpt, r221_ckpt, vgg_ckpt):\n        super().__init__(ib_ckpt, r221_ckpt)\n        self.vgg = VggFace().to(DEVICE).eval()\n        self.vgg.load_state_dict(torch.load(vgg_ckpt, map_location=DEVICE))\n        from speechbrain.inference.speaker import EncoderClassifier\n        from speechbrain.utils.fetching import LocalStrategy\n        self.btc = EncoderClassifier.from_hparams(source=\'yangwang825/ecapa-tdnn-vox2\', savedir=str(Path(vgg_ckpt).parent / \'sb_btc\'),\n                                                  run_opts={\'device\': \'cuda:0\' if DEVICE.type == \'cuda\' else \'cpu\'}, local_strategy=LocalStrategy.COPY)\n\n    @torch.inference_mode()\n    def face_features(self, paths):\n        ib, st = self.faces(paths)\n        vg = []\n        for lo in range(0, len(paths), 32):\n            xs = []\n            for p in paths[lo:lo + 32]:\n                try:\n                    xs.append(vgg_input(Image.open(p)))\n                except Exception:\n                    xs.append(np.zeros((3, 224, 224), np.float32))\n            vg.append(self.vgg(torch.tensor(np.stack(xs), device=DEVICE)).cpu().numpy())\n        vg = np.concatenate(vg).astype(np.float32)\n        vg[[s != \'ok\' for s in st]] = np.nan\n        return vg, ib, st\n\n    @torch.inference_mode()\n    def voice_features(self, paths):\n        ib, r2, st = self.voices(paths)\n        btc = []\n        for p, s in zip(paths, st):\n            w, _ = load_wave(p) if s != \'fallback_silent_or_empty\' and not s.startswith(\'fallback_unreadable\') else (None, None)\n            btc.append(self.btc.encode_batch(torch.tensor(w)[None]).squeeze().float().cpu().numpy() if w is not None\n                       else np.full(192, np.nan, np.float32))\n        return np.asarray(btc, np.float32), ib, r2, st\ndef _mount_root(base, p):\n    """Kaggle mounts a dataset at <base>/<slug> or <base>/datasets/<owner>/<slug> (newer layout)."""\n    rel = Path(p).relative_to(base).parts\n    n = 3 if rel[0] == \'datasets\' else 2 if rel[0] == \'competitions\' else 1\n    return Path(base).joinpath(*rel[:n])\ndef locate_roots(base, data_root=None, ext_root=None, max_depth=7):\n    """Find the FLAG dataset (train_English.txt next to a faces/ folder) and the MAV-Celeb v1/v2 dataset without\n    descending into any faces/ or voices/ folder: a recursive search of /kaggle/input with the 28.9 GB external tree\n    attached costs ~30 min per lookup."""\n    base = Path(base); lists, ext = [], []\n    for d, dirs, files in os.walk(base):\n        depth = len(Path(d).relative_to(base).parts)\n        if \'train_English.txt\' in files and Path(d, \'faces\').is_dir():\n            lists.append(Path(d) / \'train_English.txt\')\n        if Path(d).name in (\'v1\', \'v2\') and {\'faces\', \'voices\'} <= set(dirs):\n            ext.append(Path(d))\n        dirs[:] = [x for x in dirs if x not in (\'faces\', \'voices\')] if depth < max_depth else []\n    if data_root is None:\n        roots = sorted({_mount_root(base, p) for p in lists})\n        if len(roots) != 1:\n            raise ValueError(f\'Expected one FLAG dataset with raw media under {base}, found {roots}; set DATA_ROOT.\')\n        data_root = roots[0]\n    if ext_root is None:\n        roots = sorted({_mount_root(base, p) for p in ext})\n        if not roots:\n            raise ValueError(f\'No MAV-Celeb v1/v2 tree under {base}; attach chnnhnnguyn/external-data-mavceleb or set EXT_ROOT.\')\n        ext_root = roots[0] if len(roots) == 1 else Path(os.path.commonpath([str(r) for r in roots]))\n    if Path(data_root) in Path(ext_root).parents or Path(ext_root) == Path(data_root):\n        raise ValueError(\'DATA_ROOT must not contain the external dataset (every file lookup would scan 28.9 GB).\')\n    return Path(data_root), Path(ext_root)\nVERSION = "RF-L1-003-v1"\nVERSION = "RF-L1-004-v1"\nAGE_MID = np.array([1, 6, 15, 25, 35, 45, 55, 65, 75], np.float32)\ndef _md5(p):\n    try:\n        return hashlib.md5(Path(p).read_bytes()).hexdigest()\n    except Exception:\n        return None\ndef _cached_map(paths, fn, dim, batch=16):\n    """Embed unique files once, in batches; unreadable -> NaN row (filled with a train constant later)."""\n    keys = [_md5(p) for p in paths]\n    uniq = {}\n    for p, k in zip(paths, keys):\n        if k is not None and k not in uniq:\n            uniq[k] = p\n    items = list(uniq.items()); emb = {}\n    for lo in range(0, len(items), batch):\n        chunk = items[lo:lo + batch]\n        out = fn([p for _, p in chunk])\n        for (k, _), e in zip(chunk, out):\n            emb[k] = e\n        if (lo // batch) % 50 == 0:\n            print(f\'    {lo + len(chunk)}/{len(items)}\', flush=True)\n    nan = np.full(dim, np.nan, np.float32)\n    X = np.stack([emb.get(k, nan) if k is not None else nan for k in keys]).astype(np.float32)\n    return X\nHF_REV = {\'openai/whisper-large-v3\': \'06f233fe06e710322aca913c1bc4249a0d71fce1\', \'nateraw/vit-age-classifier\': \'4b5acc345d9692c3ff4f45d250531afd3a3da0ef\',\n          \'audeering/wav2vec2-large-robust-24-ft-age-gender\': \'bad2737f529168bc3d3fcd97edfaba9583c7d36d\',\n          \'openai/clip-vit-large-patch14\': \'32bd64288804d66eefd0ccbe215aa642df71cc41\'}\nclass AgeExtractor:\n    """Face: nateraw/vit-age-classifier (9 age groups -> expected age + probabilities).\n    Voice: audeering/wav2vec2-large-robust-24-ft-age-gender (age 0..1 x 100, gender female/male/child)."""\n\n    def __init__(self):\n        import torch.nn as nn\n        from transformers import AutoImageProcessor, ViTForImageClassification, Wav2Vec2FeatureExtractor, Wav2Vec2Model, Wav2Vec2PreTrainedModel\n        fn = \'nateraw/vit-age-classifier\'\n        self.fp = AutoImageProcessor.from_pretrained(fn, revision=HF_REV[fn])\n        self.fm = ViTForImageClassification.from_pretrained(fn, revision=HF_REV[fn]).to(DEVICE).eval()\n\n        class Head(nn.Module):\n            def __init__(self, c, n):\n                super().__init__()\n                self.dense = nn.Linear(c.hidden_size, c.hidden_size); self.dropout = nn.Dropout(c.final_dropout)\n                self.out_proj = nn.Linear(c.hidden_size, n)\n\n            def forward(self, x):\n                return self.out_proj(self.dropout(torch.tanh(self.dense(self.dropout(x)))))\n\n        class AgeGender(Wav2Vec2PreTrainedModel):  # model card of audeering/wav2vec2-large-robust-24-ft-age-gender\n            def __init__(self, c):\n                super().__init__(c)\n                self.wav2vec2 = Wav2Vec2Model(c); self.age = Head(c, 1); self.gender = Head(c, 3)\n                self.post_init()\n\n            def forward(self, x):\n                h = self.wav2vec2(x)[0].mean(1)\n                return self.age(h), torch.softmax(self.gender(h), 1)\n\n        name = \'audeering/wav2vec2-large-robust-24-ft-age-gender\'\n        self.vp = Wav2Vec2FeatureExtractor.from_pretrained(name, revision=HF_REV[name])\n        self.vm = AgeGender.from_pretrained(name, revision=HF_REV[name]).to(DEVICE).eval()\n\n    @torch.inference_mode()\n    def faces(self, paths):\n        def fn(ps):\n            out = []\n            for p in ps:\n                try:\n                    x = self.fp(Image.open(p).convert(\'RGB\'), return_tensors=\'pt\')[\'pixel_values\'].to(DEVICE)\n                    pr = torch.softmax(self.fm(x).logits, -1).float().cpu().numpy()[0]\n                    out.append(np.r_[pr @ AGE_MID, pr])\n                except Exception:\n                    out.append(np.full(10, np.nan, np.float32))\n            return out\n        return _cached_map(paths, fn, 10)\n\n    @torch.inference_mode()\n    def voices(self, paths):\n        def fn(ps):\n            out = []\n            for p in ps:\n                w, st = load_wave(p)\n                if w is None:\n                    out.append(np.full(4, np.nan, np.float32)); continue\n                x = self.vp(w, sampling_rate=16000, return_tensors=\'pt\')[\'input_values\'].to(DEVICE)\n                a, g = self.vm(x)\n                out.append(np.r_[100 * a.float().cpu().numpy()[0], g.float().cpu().numpy()[0]])  # age, p(female), p(male), p(child)\n            return out\n        return _cached_map(paths, fn, 4)\nCOND_L4 = (\'p_face_male\', \'p_voice_male\', \'gender_agree\', \'protocol_gender\')\ndef fuser_matrix4(s, streams):\n    cond = np.stack([s[k] for k in COND_L4], 1)\n    st = np.stack([s[k] for k in streams], 1)\n    return np.hstack([st, cond, (st[:, :, None] * cond[:, None, :]).reshape(len(st), -1)])\nVERSION = "RF-L2-001-v1"\nclass VB2Extractor:\n    """VoxBlink2 SimAM-ResNet100 (WeSpeaker, fine-tuned on VoxCeleb), 256-d, same fbank front end as ResNet221."""\n\n    def __init__(self, ckpt):\n        m = SimAM_ResNet100_ASP(in_planes=64, embed_dim=256, acoustic_dim=80)\n        sd = torch.load(ckpt, map_location=\'cpu\'); sd = sd.get(\'model\', sd)\n        res = m.load_state_dict(sd, strict=False)\n        if res.missing_keys or set(res.unexpected_keys) - {\'projection.weight\'}:\n            raise ValueError(f\'VoxBlink2 SimAM-ResNet100 checkpoint mismatch: {res}\')\n        self.m = m.to(DEVICE).eval()\n\n    @torch.inference_mode()\n    def voices(self, paths):\n        def fn(ps):\n            out = []\n            for p in ps:\n                w, st = load_wave(p)\n                out.append(np.full(256, np.nan, np.float32) if w is None else\n                           self.m(kaldi_fbank(w)[None].to(DEVICE)).squeeze(0).float().cpu().numpy())\n            return out\n        return _cached_map(paths, fn, 256)\nVERSION = "RF-L1-006-v1"\nVERSION = "RF-L2-002-v1"\nclass LoRAParam(nn.Module):\n    def __init__(self, out_f, in_f, rank, alpha):\n        super().__init__()\n        self.A = nn.Parameter(torch.randn(rank, in_f, dtype=torch.float32) / rank)\n        self.B = nn.Parameter(torch.zeros(out_f, rank, dtype=torch.float32))\n        self.scale = alpha / rank\n\n    def forward(self, W):\n        return W + (self.B @ self.A).to(W.dtype) * self.scale\ndef add_lora(ib, MT, rank, alpha):\n    import torch.nn.utils.parametrize as P\n    for p in ib.parameters():\n        p.requires_grad_(False)\n    params = []\n    for name, m in ib.modality_trunks[MT.AUDIO].named_modules():\n        if isinstance(m, nn.MultiheadAttention):\n            for mod, attr in ((m, \'in_proj_weight\'), (m.out_proj, \'weight\')):\n                lp = LoRAParam(*getattr(mod, attr).shape, rank, alpha).to(DEVICE)\n                P.register_parametrization(mod, attr, lp)\n                params += [lp.A, lp.B]\n    if not params:\n        raise ValueError(\'no attention module found in the ImageBind audio trunk\')\n    return params\nVERSION = "RF-L2-003b-v1"\n@torch.inference_mode()\ndef embed_audio(ib, MT, paths):\n    def fn(ps):\n        res = []\n        for p in ps:\n            w, st = load_wave(p)\n            if w is None: res.append(np.full(1024, np.nan, np.float32)); continue\n            res.append(ib({MT.AUDIO: ib_audio_clips(w)[None].half().to(DEVICE)})[MT.AUDIO].float().cpu().numpy()[0])\n        return res\n    return _cached_map(paths, fn, 1024)\nVERSION = "RF-L2-006-v1"\nCLIP_NAME = \'openai/clip-vit-large-patch14\'\nclass ClipFaceExtractor:\n    def __init__(self, name=CLIP_NAME):\n        from transformers import CLIPImageProcessor, CLIPVisionModelWithProjection\n        self.p = CLIPImageProcessor.from_pretrained(name, revision=HF_REV.get(name))\n        self.m = CLIPVisionModelWithProjection.from_pretrained(name, revision=HF_REV.get(name)).to(DEVICE).eval()   # fp32: per-image results\n\n    @torch.inference_mode()\n    def faces(self, paths):\n        def fn(ps):\n            out = []\n            for p in ps:   # one image at a time: the embedding of a file never depends on other files\n                try:\n                    x = self.p(images=Image.open(p).convert(\'RGB\'), return_tensors=\'pt\')[\'pixel_values\'].to(DEVICE)\n                    out.append(self.m(pixel_values=x).image_embeds[0].float().cpu().numpy())\n                except Exception:\n                    out.append(np.full(768, np.nan, np.float32))\n            return out\n        return _cached_map(paths, fn, 768)\ndef vb2_voices(ex, paths, max_sec=60):\n    """VB2Extractor on the first max_sec seconds (same window as the language ID)."""\n    def fn(ps):\n        out = []\n        for p in ps:\n            w, st = load_wave(p)\n            out.append(np.full(256, np.nan, np.float32) if w is None else\n                       ex.m(kaldi_fbank(w[:16000 * max_sec])[None].to(DEVICE)).squeeze(0).float().cpu().numpy())\n        return out\n    with torch.inference_mode():\n        return _cached_map(paths, fn, 256)\nVERSION = "RF-L2-008-v1"\nVERSION = "RF-L2-009-v1"\nimport threading as _threading\nVERSION = "RF-E2E-v1"\nclass ProjQ(nn.Module):\n    def __init__(self, df, dv, c):\n        super().__init__()\n        self.f = nn.Sequential(nn.LayerNorm(df), nn.Dropout(c[\'drop_f\']), nn.Linear(df, c[\'dim\']))\n        self.v = nn.Sequential(nn.LayerNorm(dv), nn.Dropout(c[\'drop_v\']), nn.Linear(dv, c[\'dim\']))\ndef _u(X):\n    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)\n@torch.no_grad()\ndef t3_scores(models, clip, ibf, sam, r2, btc, ibv=None, mu_ib=None, ib_scale=0.5, vextra=()):\n    """ibv given (RF-L2-022 projector QI): the raw ImageBind voice joins the voice side as ib_scale * unit(unit(ibv) - mu_ib).\n    vextra (RF-L2-045): more unit-normalised voice blocks appended after ECAPA (e.g. the Whisper encoder mean)."""\n    vb = [_u(sam), _u(r2), _u(btc)] + [_u(e) for e in vextra] + ([] if ibv is None else [ib_scale * _u(_u(ibv) - mu_ib)])\n    f = torch.tensor(np.hstack([_u(clip), _u(ibf)]).astype(np.float32)); v = torch.tensor(np.hstack(vb).astype(np.float32))\n    return np.mean([(F.normalize(m.f(f), dim=-1) * F.normalize(m.v(v), dim=-1)).sum(1).numpy() for m in models], 0)\nclass WhisperVoiceExtractor:\n    name = \'openai/whisper-large-v3\'\n\n    def __init__(self, max_sec=30):\n        import transformers as T\n        self.max_sec = max_sec; self.fe = T.WhisperFeatureExtractor.from_pretrained(self.name, revision=HF_REV[self.name])\n        self.m = T.WhisperModel.from_pretrained(self.name, revision=HF_REV[self.name], torch_dtype=torch.float16).encoder.to(DEVICE).eval(); self.dim = self.m.config.d_model\n\n    def _one(self, w):\n        x = self.fe(w, sampling_rate=16000, return_tensors=\'pt\').input_features.to(DEVICE, torch.float16)\n        h = self.m(x).last_hidden_state[0]\n        n = max(1, min(h.shape[0], int(np.ceil(len(w) / 16000 / 30 * h.shape[0]))))\n        return h[:n].float().mean(0).cpu().numpy()\n\n    @torch.inference_mode()\n    def voices(self, paths):\n        def fn(ps):\n            out = []\n            for p in ps:   # one clip at a time\n                w, st = load_wave(p)\n                out.append(np.full(self.dim, np.nan, np.float32) if w is None else self._one(w[:16000 * self.max_sec]).astype(np.float32))\n            return out\n        return _cached_map(paths, fn, self.dim)\nclass E2EExtractor:\n    """All per-file embeddings the scorer needs; every file is embedded on its own (no batch padding, no file stats).\n    RF-L2-059: vx = extra t3 voice blocks from the manifest (only \'whv3\'); ages() = RF-L1-004 face / voice age (first column)."""\n\n    def __init__(self, ckpts, vx=()):\n        self.c, self.vx = ckpts, tuple(vx)\n        assert set(self.vx) <= {\'whv3\'}, self.vx\n\n    def ages(self, fpaths, vpaths):\n        ag = AgeExtractor(); a = ag.faces(fpaths)[:, 0], ag.voices(vpaths)[:, 0]; del ag; torch.cuda.empty_cache(); return a\n\n    def faces(self, paths):\n        ex = L2Extractor(self.c[\'imagebind\'], self.c[\'r221\'], self.c[\'vgg\']); vg, ibf, _ = ex.face_features(paths); del ex\n        torch.cuda.empty_cache(); cl = ClipFaceExtractor(); clip = cl.faces(paths); del cl; torch.cuda.empty_cache()\n        return dict(xf=vg, ibf=ibf, clip=clip)\n\n    def voices(self, paths):\n        ex = L2Extractor(self.c[\'imagebind\'], self.c[\'r221\'], self.c[\'vgg\']); btc, ibv, r2, _ = ex.voice_features(paths); del ex\n        torch.cuda.empty_cache()\n        ib, MT = load_imagebind(self.c[\'imagebind\']); params = add_lora(ib, MT, 8, 16)\n        state = torch.load(self.c[\'lora\'], map_location=DEVICE)\n        for p, v in zip(params, state[\'params\']): p.data.copy_(v)\n        lora = embed_audio(ib, MT, paths); del ib, params; torch.cuda.empty_cache()\n        vb = VB2Extractor(self.c[\'sam\']); sam = vb2_voices(vb, paths, 60); del vb; torch.cuda.empty_cache()\n        out = dict(xv=btc, ibv=ibv, r221=r2, lora=lora, sam=sam)\n        if \'whv3\' in self.vx:\n            wx = WhisperVoiceExtractor(); out[\'whv3\'] = wx.voices(paths); del wx; torch.cuda.empty_cache()\n        return out\n\n\n# =========================================================================== RF-FINAL: the submitted system (RF-L2-059)\n# Every pair is scored on its own from fixed parameters learned on the training data (final_params.npz, written by\n# make_final_release.py); nothing is fitted, normalised or looked up across pairs at test time.\n#   t3   cosine of the QV-Whisper projector: face [CLIP ViT-L/14 || ImageBind] x voice [SimAM-ResNet100 || ResNet221 || ECAPA || Whisper]\n#   ib   ImageBind face x ImageBind audio with LoRA C1b, each centred by its training mean\n#   age  -|ViT-age(face) - b(audeering age(voice))|, no_gender files only\n#   conditions: P(male) of the face (VGG-Face) and of the voice (ECAPA) from logistic probes, their agreement, the file\'s protocol\n#   nf, nv (RF-L2-092, when the manifest lists them): log norm of the projected face / voice embedding before L2 normalisation\n#        (mean over the projector seeds), a per-input property of each file; no cohort, no test statistic\n# gender files: logistic regression on (t3, ib[, nf, nv]); no_gender files: logistic regression on streams + conditions + products.\nVERSION = "RF-FINAL-v2"\n\n\n@torch.no_grad()\ndef qf_norms(models, clip, ibf, sam, r2, btc, vextra=()):\n    """RF-L2-092: per file, mean over the projector seeds of log ||projection|| before L2 normalisation (face side, voice side)."""\n    f = torch.tensor(np.hstack([_u(clip), _u(ibf)]).astype(np.float32)); v = torch.tensor(np.hstack([_u(sam), _u(r2), _u(btc)] + [_u(e) for e in vextra]).astype(np.float32))\n    return (np.mean([torch.log(m.f(f).norm(dim=1)).numpy() for m in models], 0), np.mean([torch.log(m.v(v).norm(dim=1)).numpy() for m in models], 0))\n\n\nclass FinalScorer:\n    def __init__(self, art):\n        art = Path(art); self.man = read_json(art / \'manifest_final.json\')\n        with np.load(art / \'final_params.npz\') as z:\n            self.p = {k: z[k] for k in z.files}\n        self.vx = [b[\'block\'] for b in self.man[\'t3_voice_extra\']]\n        dv = 256 + 256 + 192 + sum(int(b[\'dim\']) for b in self.man[\'t3_voice_extra\'])\n        self.proj = []\n        for sd in torch.load(art / \'projector_Q.pt\', map_location=\'cpu\'):\n            m = ProjQ(768 + 1024, dv, self.man[\'projector\']); m.load_state_dict(sd); self.proj.append(m.eval())\n\n    def _lr(self, name, X):   # logistic regression decision values (fused scaler + coefficients)\n        p = self.p; return ((X - p[f\'{name}_mean\']) / p[f\'{name}_scale\']) @ p[f\'{name}_coef\'] + p[f\'{name}_b\']\n\n    def score(self, x, proto):\n        """x: per-pair rows (xf VGG-Face, ibf, clip, age_f; xv ECAPA, lora, r221, sam, whv3, age_v). Returns lower-is-same."""\n        p = self.p; fill = lambda k, X: fill_nonfinite(X, p[f\'fill_{k}\'])[0]\n        xf, xv, ibf, r2 = fill(\'xf\', x[\'xf\']), fill(\'xv\', x[\'xv\']), fill(\'ib_face\', x[\'ibf\']), fill(\'r221\', x[\'r221\'])\n        cl, sam, vext = fill(\'clip\', x[\'clip\']), fill(\'sam\', x[\'sam\']), [fill(b, x[b]) for b in self.vx]\n        s = {\'t3\': t3_scores(self.proj, cl, ibf, sam, r2, xv, vextra=vext),\n             \'ib\': (_unit(ibf - p[\'mu_ibf\']) * _unit(fill(\'lora\', x[\'lora\']) - p[\'fill_lora\'])).sum(1)}\n        if \'nf\' in self.man[\'streams_g\'] or \'nf\' in self.man[\'streams_ng\']:   # RF-L2-092\n            s[\'nf\'], s[\'nv\'] = qf_norms(self.proj, cl, ibf, sam, r2, xv, vext)\n        pf = 1 / (1 + np.exp(-(((xf - p[\'prep_mf\']) / p[\'prep_sf\']) @ p[\'prep_P\']).astype(np.float32) @ p[\'gf_coef\'] - p[\'gf_b\']))\n        pv = 1 / (1 + np.exp(-((xv - p[\'prep_mv\']) / p[\'prep_sv\']).astype(np.float32) @ p[\'gv_coef\'] - p[\'gv_b\']))\n        s.update(p_face_male=pf, p_voice_male=pv, gender_agree=pf * pv + (1 - pf) * (1 - pv), protocol_gender=np.full(len(xf), float(proto == \'gender\')))\n        if proto == \'gender\':   # the file\'s protocol label set by the organisers (declared)\n            out = -self._lr(\'fg\', np.stack([s[k] for k in self.man[\'streams_g\']], 1))\n        else:\n            a_f = np.where(np.isfinite(x[\'age_f\']), x[\'age_f\'], p[\'age_fill\'][0]); a_v = np.where(np.isfinite(x[\'age_v\']), x[\'age_v\'], p[\'age_fill\'][1])\n            s[\'age\'] = -np.abs(a_f - np.polyval(p[\'age_b\'], a_v))\n            out = -self._lr(\'fn\', fuser_matrix4(s, self.man[\'streams_ng\']))\n        if not np.isfinite(out).all(): raise ValueError(\'non-finite scores\')\n        return out\n'

CODE_DIR = OUT/'code'; CODE_DIR.mkdir(exist_ok=True)
module_path = CODE_DIR/'flag_e2e_core.py'
module_path.write_bytes(CORE_SOURCE.encode('utf-8'))
spec = importlib.util.spec_from_file_location('flag_e2e_core', module_path)
L = importlib.util.module_from_spec(spec); sys.modules['flag_e2e_core'] = L; spec.loader.exec_module(L)
import numpy as np, pandas as pd, torch
from IPython.display import FileLink, display
from scipy.stats import spearmanr
print('Device:', L.DEVICE, '| Code SHA256:', L.file_hash(module_path))
if ART is None:
    ART = next((p.parent for p in Path('/kaggle/input').rglob('manifest_final.json')), None)
if ART is None: raise FileNotFoundError('Thiếu dataset artifact (manifest_final.json): Add Input.')
ART = Path(ART); man = json.loads((ART/'manifest_final.json').read_text())
bad = [n for n, h in man['files'].items() if (ART/n).exists() and L.file_hash(ART/n) != h]
if bad: raise ValueError(f'Artifact khác manifest: {bad}')
if REF_ZIP is None:   # dev reference shipped with the artifacts; Kaggle unpacks a .zip inside a dataset into a folder
    REF_ZIP = next(iter(sorted(ART.glob('submission_*.zip'))), None) or next((p for p in sorted(ART.glob('submission_*')) if p.is_dir()), None)
DATA_ROOT, _ = L.locate_roots(Path('/kaggle/input'), DATA_ROOT, ART)   # the artifact folder (ext.npz) stands in for the external tree
models = OUT/'models'; models.mkdir(exist_ok=True)
def fetch(path, name):   # a copy shipped in the artifact dataset wins; otherwise download (4 tries)
    t = Path(path) if path is not None else models/f'{name}.pt'
    if path is None and not t.exists() and (ART/f'{name}.pt').exists():
        t.symlink_to(ART/f'{name}.pt')   # the dataset is read-only; SpeechBrain writes its files next to the VGG checkpoint
    for k in range(4):
        if t.exists(): break
        try:
            print('Tải', name, flush=True); urllib.request.urlretrieve(URL[name], t.with_suffix('.part')); t.with_suffix('.part').replace(t)
        except OSError as e:
            print('  lỗi tải', name, type(e).__name__, '- thử lại', flush=True); t.with_suffix('.part').unlink(missing_ok=True); time.sleep(30 * (k + 1))
    assert L.file_hash(t) == SHA[name], f'{name}: SHA-256 khác bản đã kiểm'
    return t
CK = dict(imagebind=fetch(IB_CKPT, 'imagebind_huge'), r221=fetch(R221_CKPT, 'resnet221_lm'), vgg=fetch(VGG_CKPT, 'vgg_face'),
          sam=fetch(VB2_CKPT, 'voxblink2_samresnet100_ft'), lora=ART/'lora_C1.pt')
print('Data:', DATA_ROOT, '| artifacts:', ART, '| reference ZIP:', REF_ZIP)

## 1. Bộ chấm cố định (chỉ nạp tham số đã học từ artifact)

In [ ]:
print("[cell] scorer", flush=True)
def make_xscorer(L):
    """FinalScorer of the core with the projector input widened by the FACTOR-01 blocks listed in manifest['add']
    ({'face': [[name, dim], ...], 'voice': [...]}, appended after the core blocks, each unit-normalised). Same fills, ib / gender /
    age streams and fusers as L.FinalScorer.score; only t3 / nf / nv read the wider input. No 'add' -> L.FinalScorer unchanged."""
    import numpy as np, torch, torch.nn.functional as F
    from pathlib import Path

    def _u(X):
        X = np.nan_to_num(np.asarray(X, np.float32)); return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)

    class XScorer(L.FinalScorer):
        def __init__(self, art):
            art = Path(art); self.man = L.read_json(art / 'manifest_final.json')
            with np.load(art / 'final_params.npz') as z:
                self.p = {k: z[k] for k in z.files}
            self.vx = [b['block'] for b in self.man['t3_voice_extra']]
            add = self.man.get('add') or {}
            self.xf = [k for k, _ in add.get('face', [])]; self.xv = [k for k, _ in add.get('voice', [])]
            df = 768 + 1024 + sum(int(d) for _, d in add.get('face', []))
            dv = 256 + 256 + 192 + sum(int(b['dim']) for b in self.man['t3_voice_extra']) + sum(int(d) for _, d in add.get('voice', []))
            self.proj = []
            for sd in torch.load(art / 'projector_Q.pt', map_location='cpu'):
                m = L.ProjQ(df, dv, self.man['projector']); m.load_state_dict(sd); self.proj.append(m.eval())

        def score(self, x, proto):
            if not (self.xf or self.xv):
                return L.FinalScorer.score(self, x, proto)
            p = self.p; fill = lambda k, X: L.fill_nonfinite(X, p[f'fill_{k}'])[0]
            for k in self.xf + self.xv:
                assert np.isfinite(np.asarray(x[k], np.float32)).all(), f'extra block {k}: non-finite rows'
            xf, xv, ibf, r2 = fill('xf', x['xf']), fill('xv', x['xv']), fill('ib_face', x['ibf']), fill('r221', x['r221'])
            cl, sam, vext = fill('clip', x['clip']), fill('sam', x['sam']), [fill(b, x[b]) for b in self.vx]
            f = torch.tensor(np.hstack([_u(cl), _u(ibf)] + [_u(x[k]) for k in self.xf]))
            v = torch.tensor(np.hstack([_u(sam), _u(r2), _u(xv)] + [_u(e) for e in vext] + [_u(x[k]) for k in self.xv]))
            with torch.no_grad():
                s = {'t3': np.mean([(F.normalize(m.f(f), dim=-1) * F.normalize(m.v(v), dim=-1)).sum(1).numpy() for m in self.proj], 0),
                     'nf': np.mean([torch.log(m.f(f).norm(dim=1)).numpy() for m in self.proj], 0),
                     'nv': np.mean([torch.log(m.v(v).norm(dim=1)).numpy() for m in self.proj], 0)}
            s['ib'] = (L._unit(ibf - p['mu_ibf']) * L._unit(fill('lora', x['lora']) - p['fill_lora'])).sum(1)
            pf = 1 / (1 + np.exp(-(((xf - p['prep_mf']) / p['prep_sf']) @ p['prep_P']).astype(np.float32) @ p['gf_coef'] - p['gf_b']))
            pv = 1 / (1 + np.exp(-((xv - p['prep_mv']) / p['prep_sv']).astype(np.float32) @ p['gv_coef'] - p['gv_b']))
            s.update(p_face_male=pf, p_voice_male=pv, gender_agree=pf * pv + (1 - pf) * (1 - pv), protocol_gender=np.full(len(xf), float(proto == 'gender')))
            if proto == 'gender':
                out = -self._lr('fg', np.stack([s[k] for k in self.man['streams_g']], 1))
            else:
                a_f = np.where(np.isfinite(x['age_f']), x['age_f'], p['age_fill'][0]); a_v = np.where(np.isfinite(x['age_v']), x['age_v'], p['age_fill'][1])
                s['age'] = -np.abs(a_f - np.polyval(p['age_b'], a_v))
                out = -self._lr('fn', L.fuser_matrix4(s, self.man['streams_ng']))
            if not np.isfinite(out).all(): raise ValueError('non-finite scores')
            return out
    return XScorer

scorer = make_xscorer(L)(ART)   # FACTOR-01: wider projector input when manifest has 'add'
print('FACTOR-01 extra blocks:', scorer.xf, scorer.xv, flush=True)   # fixed parameters only; no training data is read
CELLS = NAMES or L.CELLS

## 2. Danh sách cặp, trích embedding từng file (mỗi file một mình)

In [ ]:
print("[cell] pairs+extract", flush=True)
lists = {}
for (proto, lang), member in CELLS.items():
    lp = L.dev_list_path(DATA_ROOT, proto, lang); t = L.table(lp, ['pair_id', 'voice', 'face'])
    fp, vp = L.media_paths(lp, t.face), L.media_paths(lp, t.voice)
    csv = None
    if USE_CSV:
        try:
            xa, xb, t2, _ = L.load_dev_lenient(DATA_ROOT, proto, lang, scorer.man['dimensions'], False, scorer.p['fill_xf'], scorer.p['fill_xv'])
            assert (t2.pair_id.values == t.pair_id.values).all(); csv = (xa, xb)
        except Exception as ex:
            print(f'  {proto}/{lang}: không dùng CSV ({type(ex).__name__}), tự trích VGG / ECAPA', flush=True)
    n = len(t) if SMOKE_PAIRS is None else min(SMOKE_PAIRS, len(t))
    lists[(proto, lang)] = dict(t=t.head(n), fp=fp[:n], vp=vp[:n], csv=None if csv is None else (csv[0][:n], csv[1][:n]))
    print(f'  {proto}/{lang}: {n} cặp', flush=True)
ufp = sorted({p for d in lists.values() for p in d['fp']}); uvp = sorted({p for d in lists.values() for p in d['vp']})
ex = L.E2EExtractor(CK, scorer.vx)
Fz = ex.faces(ufp); Vz = ex.voices(uvp)
Fz['age'], Vz['age'] = ex.ages(ufp, uvp)   # age stream of the no_gender files
fi = {p: i for i, p in enumerate(ufp)}; vi = {p: i for i, p in enumerate(uvp)}
np.savez_compressed(OUT/'embeddings.npz', **{f'face_{k}': v for k, v in Fz.items()}, **{f'voice_{k}': v for k, v in Vz.items()})

In [ ]:
print("[cell] factor extra blocks", flush=True)
# FACTOR-01 step 3: extra blocks of each trial list from the attached kernel outputs (FLAG_65xc C5s faces, FLAG_47b rn293 / ecapa1024)
def _l2n(X):
    X = np.asarray(X, np.float32); return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)

_XFILES = None
def _one(pat):   # one walk of the input tree, skipping the media / artifact datasets (hundreds of thousands of files)
    global _XFILES
    if _XFILES is None:
        import os
        _XFILES, skip = [], ('raw-v4', 'mavceleb', 'evalphase', '-art-p')
        for r, ds, fs in os.walk(Path('/kaggle/input')):
            ds[:] = [d for d in ds if not any(s in d for s in skip)]
            _XFILES += [Path(r) / f for f in fs if f.endswith(('.npy', '.txt'))]
    hit = sorted({p.parent for p in _XFILES if p.name == pat})
    assert len(hit) == 1, f'{pat}: need exactly one folder, found {hit}'
    return hit[0]

def XTRA(proto, lang, t, fp, vp):
    if not (scorer.xf or scorer.xv):
        return {}
    name = f'{proto}_{lang}'; want = [str(p) for p in t.pair_id]; out = {}
    def rows(folder, arrs):
        idx = [l.split() for l in (folder / f'index_{name}.txt').read_text().splitlines() if l.strip()]
        pos = {r[0]: i for i, r in enumerate(idx)}
        assert set(want) <= set(pos), f'{name}: {len(set(want) - set(pos))} pairs missing in {folder}'
        sel = np.array([pos[p] for p in want])
        for j, p in enumerate(want[:50] + want[-50:]):          # media paths of the index = paths of the list (suffix)
            r = idx[pos[p]]; k = want.index(p)
            assert str(vp[k]).replace('\\', '/').endswith(r[1]) and str(fp[k]).replace('\\', '/').endswith(r[2]), (name, p, r, vp[k], fp[k])
        X = np.concatenate([_l2n(np.load(folder / f'{a}_{name}.npy'))[sel] for a in arrs], 1)
        assert len(np.load(folder / f'{arrs[0]}_{name}.npy')) == len(idx) and np.isfinite(X).all(), (name, arrs)
        return X
    for k in scorer.xf + scorer.xv:
        if k == 'f_c5s':
            out[k] = rows(_one(f'face_clipC5_{name}.npy'), ['face_clipC5'])
        elif k == 'v_rl':
            out[k] = rows(_one(f'voice_rn293_{name}.npy'), ['voice_rn293', 'voice_ecapa1024'])
        else:
            raise AssertionError(f'extra block {k}: no Evaluation source wired (FACTOR-01 step 3 wires f_c5s, v_rl)')
        print(f'  {name}: {k} {out[k].shape}', flush=True)
    return out


## 3. Chấm từng cặp, tự kiểm mức 0, đóng gói

In [ ]:
print("[cell] score+level0+zip", flush=True)
pack = OUT/'submission.zip'; scores = {}; L0D = {}
with zipfile.ZipFile(pack, 'w', zipfile.ZIP_DEFLATED) as z:
    for (proto, lang), member in CELLS.items():
        d = lists[(proto, lang)]; a_ = np.array([fi[p] for p in d['fp']]); b_ = np.array([vi[p] for p in d['vp']])
        x = dict(xf=Fz['xf'][a_], ibf=Fz['ibf'][a_], clip=Fz['clip'][a_], xv=Vz['xv'][b_], ibv=Vz['ibv'][b_], r221=Vz['r221'][b_],
                 lora=Vz['lora'][b_], sam=Vz['sam'][b_], **{b: Vz[b][b_] for b in scorer.vx}, age_f=Fz['age'][a_], age_v=Vz['age'][b_])
        if d['csv'] is not None: x['xf'], x['xv'] = d['csv']
        x.update(XTRA(proto, lang, d['t'], d['fp'], d['vp']))   # FACTOR-01 extra blocks
        s = scorer.score(x, proto)
        h = len(s) // 2; part = scorer.score({k: v[h:] for k, v in x.items()}, proto)   # level 0: a pair's score ignores the others
        L0D[member] = float(np.abs(part - s[h:]).max()); tol = 1e-4 * max(1.0, float(np.abs(s).max()))
        print(f'  level 0 {member}: max |full - half| {L0D[member]:.2e} (tol {tol:.1e}, score scale {np.abs(s).max():.2f})', flush=True)
        assert L0D[member] <= tol, 'score depends on other pairs'
        scores[member] = dict(zip(d['t'].pair_id, s))
        z.writestr(member, ''.join(f'{pid} {v:.8f}\n' for pid, v in zip(d['t'].pair_id, s)))
print('level-0 self-check OK')
chk = {}
if REF_ZIP is not None:
    ref = zipfile.ZipFile(REF_ZIP) if Path(REF_ZIP).is_file() else None
    for member, sc in scores.items():
        txt = ref.read(member).decode() if ref is not None else (Path(REF_ZIP) / member).read_text()
        r = dict(l.split() for l in txt.splitlines()); k = list(sc)
        x_ = np.array([sc[p] for p in k]); y_ = np.array([float(r[p]) for p in k])
        chk[member] = dict(pairs=len(k), spearman=round(float(spearmanr(x_, y_)[0]), 5), max_abs_diff=round(float(np.abs(x_ - y_).max()), 5))
    print(json.dumps(chk, indent=1))
L.write_json(OUT/'run_report.json', dict(version=L.VERSION, system=man['system'], code_sha256=L.file_hash(module_path), artifact_manifest=man['files'],
             checkpoints=SHA, use_csv=USE_CSV, smoke_pairs=SMOKE_PAIRS, check_vs_reference=chk, score_orientation='lower_is_same', level0=True, level0_max_abs_diff=L0D))
display(FileLink(str(pack)))